# Obligatorio Taller de Tecnologías 2

### Setup del ambiente

Instalamos modulos necesarios para el proyecto

In [1]:
!pip -q install -U \
  "langchain==0.3.27" \
  "langchain-community==0.3.27" \
  "langchain-huggingface==0.3.1" \
  "transformers==4.55.2" \
  "huggingface_hub==0.34.4" \
  "sentence-transformers" \
  "chromadb" \
  "pandas"

!pip install -q pypdf PyPDF2 langchain-text-splitters


[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: C:\Users\Emi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: C:\Users\Emi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
# Comprobar versión de Python y GPU/CPU
import sys, subprocess, torch
print(sys.version)
try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA disponible:", torch.cuda.is_available())
except Exception as e:
    print("PyTorch no disponible o sin CUDA", e)


3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
PyTorch: 2.9.0+cpu
CUDA disponible: False


### Peparamos el LLM que vamos a utilizar de HuggingFace

In [3]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
import os
HF_TOKEN = os.environ["HF_TOKEN"]
hf_endpoint = HuggingFaceEndpoint(
    repo_id=MODEL_ID,
    task="conversational",
    huggingfacehub_api_token=HF_TOKEN,
    temperature=0.7,
    max_new_tokens=256,
)

llm = ChatHuggingFace(llm=hf_endpoint)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente útil y conciso."),
    ("human", "Responde a la siguiente instrucción: {instruccion}"),
])

chain = prompt | llm | StrOutputParser()
print(chain.invoke({"instruccion": "Explica en 3 frases qué es un LLM y nombra 2 casos de uso."}))

C:\Users\Emi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Un LLM (Large Language Model) es un tipo de modelo de inteligencia artificial que ha sido entrenado en grandes volúmenes de texto para entender, generar y responder a preguntas de forma natural.  
Dos casos de uso son la redacción de contenido automatizada y la asistencia en tareas de traducción o resolución de problemas.


### Preparamos embeddings


In [4]:

from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

# Probar los embeddings
test_text = "Este es un texto de prueba"
test_embedding = embeddings.embed_query(test_text)
print(f"Dimensión del embedding: {len(test_embedding)}")
print(f"Primeros 5 valores: {test_embedding[:5]}")


Dimensión del embedding: 384
Primeros 5 valores: [-0.04650536924600601, 0.07088886201381683, -0.07311516255140305, -0.011487394571304321, 0.03767696022987366]


### Creamos la base de datos vectorial para almacenar los embeddings

Cargamos el dataset de kaggle

In [5]:
import pandas as pd

df = pd.read_csv('Datasets de kaggle/netflix_titles.csv')

# Mostramos un poco de la informacion del dataset
print(f"Total de registros: {len(df)}")
print(f"\nColumnas disponibles:")
print(df.columns.tolist())
print(f"\nPrimeras filas:")
print(df.head())
print(f"\nInformación del dataset:")
print(df.info())

Total de registros: 8807

Columnas disponibles:
['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'rating', 'duration', 'listed_in', 'description']

Primeras filas:
  show_id     type                  title         director  \
0      s1    Movie   Dick Johnson Is Dead  Kirsten Johnson   
1      s2  TV Show          Blood & Water              NaN   
2      s3  TV Show              Ganglands  Julien Leclercq   
3      s4  TV Show  Jailbirds New Orleans              NaN   
4      s5  TV Show           Kota Factory              NaN   

                                                cast        country  \
0                                                NaN  United States   
1  Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...   South Africa   
2  Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...            NaN   
3                                                NaN            NaN   
4  Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...          India

### Creamos documentos para el vector store

In [6]:
from langchain.schema import Document

documents = []
for idx, row in df.iterrows():
    # Combinamos campos relevantes
    content = f"Título: {row['title']}\n"
    content += f"Tipo: {row['type']}\n"
    content += f"Director: {row.get('director', 'N/A')}\n"
    content += f"Elenco: {row.get('cast', 'N/A')}\n"
    content += f"País: {row.get('country', 'N/A')}\n"
    content += f"Año: {row.get('release_year', 'N/A')}\n"
    content += f"Categoría: {row.get('listed_in', 'N/A')}\n"
    content += f"Descripción: {row.get('description', 'N/A')}\n"
    
    doc = Document(
        page_content=content,
        metadata={
            'title': row['title'],
            'type': row['type'],
            'year': row.get('release_year', 'N/A'),
            'source': 'netflix'
        }
    )
    documents.append(doc)

print(f"Total de documentos creados: {len(documents)}")

Total de documentos creados: 8807


### Creamos el vectore store con ChromaDB

In [7]:
from langchain_community.vectorstores import Chroma
import os
import shutil

# Directorio donde va a persistir la base de datos
persist_directory = './chroma_db'

# Paso agregado para limpiar la base de datos existente (explicado en la documentacion)
if os.path.exists(persist_directory):
    print(f"Limpiando base de datos existente en {persist_directory}...")
    shutil.rmtree(persist_directory)

print("\nCreando base de datos vectorial...")

# Implementacion de ids para evitar duplicados (tambien explicado en la documentacion)
ids = [row['show_id'] for idx, row in df.iterrows()]

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    ids=ids,
    persist_directory=persist_directory
)

print("\nBase de datos creada.")

Limpiando base de datos existente en ./chroma_db...

Creando base de datos vectorial...

Base de datos creada.


### Prueba de busqueda semantica


In [8]:
# Hacer una consulta de prueba
query = "películas de ciencia ficción con viajes en el tiempo"
resultados = vectorstore.similarity_search(query, k=3)

print(f"\nResultados para: '{query}'")

for i, doc in enumerate(resultados, 1):
    print(f"\n{i}. {doc.metadata['title']} ({doc.metadata['year']})")
    print(f"\n{doc.page_content[:200]}...")


Resultados para: 'películas de ciencia ficción con viajes en el tiempo'

1. The Time Machine (2002)

Título: The Time Machine
Tipo: Movie
Director: Simon Wells
Elenco: Guy Pearce, Samantha Mumba, Jeremy Irons, Mark Addy, Sienna Guillory, Phyllida Law, Alan Young, Omero Mumba, Yancey Arias, Orlando Jo...

2. See You Yesterday (2019)

Título: See You Yesterday
Tipo: Movie
Director: Stefon Bristol
Elenco: Eden Duncan-Smith, Dante Crichlow, Astro, Johnathan Nieves, Marsha Stephanie Blake, Wavyy Jonez, Myra Lucretia Taylor
País: Unite...

3. Time Trap (2017)

Título: Time Trap
Tipo: Movie
Director: Mark Dennis, Ben Foster
Elenco: Andrew Wilson, Cassidy Gifford, Brianne Howey, Reiley McClendon, Olivia Draguicevich, Max Wright, Hans Marrero, Rich Skidmore, C...


### Configuramos el retriever básico

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}  # Recuperar los 3 (k) documentos más relevantes
)

# Probamos si se recuperan documentos
docs = retriever.invoke("comedias románticas")
print(f"\nSe recuperaron {len(docs)} documentos")


Se recuperaron 3 documentos


# AGENTE RAG

#### Creamos el agente RAG

In [10]:
from langchain.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
import json

def format_docs(docs):
    """Formatea una lista de documentos en texto"""
    return "\n\n".join([doc.page_content for doc in docs])

# Definimos las tools que le vamos a asociar al agente
@tool
def search_netflix(query: str) -> str:
    """Busca películas o series en la base de datos de Netflix según criterios como género, actor, año, etc."""
    docs = retriever.invoke(query)
    if not docs:
        return "No se encontraron resultados para tu búsqueda."
    
    results = []
    for i, doc in enumerate(docs[:3], 1):
        results.append(f"{i}. {doc.metadata['title']} ({doc.metadata['year']})")
    
    details = "\n\n".join([doc.page_content[:300] + "..." for doc in docs[:3]])
    return "Resultados encontrados:\n" + "\n".join(results) + "\n\nDetalles:\n" + details

@tool
def get_movie_details(title: str) -> str:
    """Obtiene información detallada de una película o serie específica cuando conoces su título exacto."""
    docs = vectorstore.similarity_search(f"título: {title}", k=1)
    if not docs:
        return f"No se encontró información sobre '{title}'"
    return docs[0].page_content

@tool  
def count_titles_by_type(content_type: str) -> str:
    """Cuenta cuántas películas o series hay en Netflix."""
    if content_type.lower() == "all":
        return f"Total de títulos en Netflix: {len(documents)}"
    count = sum(1 for doc in documents if doc.metadata.get('type', '').lower() == content_type.lower())
    return f"Hay {count} {content_type}s en Netflix"

tools = [search_netflix, get_movie_details, count_titles_by_type]


# Creamos el agente con memoria
class AgenticRAGWithMemory:
    def __init__(self, llm, tools, embeddings, vectorstore, max_history=5):
        self.llm = llm
        self.tools = {tool.name: tool for tool in tools}
        self.embeddings = embeddings
        self.vectorstore = vectorstore
        self.conversation_history = []
        self.max_history = max_history  # Limitamos memoria a últimos 5 intercambios
        
    def _format_tools(self) -> str:
        """Formatea las herramientas para el prompt"""
        tool_descriptions = []
        for name, tool in self.tools.items():
            tool_descriptions.append(f"- {name}: {tool.description}")
        return "\n".join(tool_descriptions)
    
    def _get_conversation_context(self) -> str:
        """Obtiene el contexto de la conversación reciente"""
        if not self.conversation_history:
            return "Esta es la primera interacción."
        
        # Toma solo los ultimos max_history intercambios
        recent_history = self.conversation_history[-self.max_history:]
        
        context = "Historial de conversación:\n"
        for msg in recent_history:
            role = "Usuario" if msg["role"] == "user" else "Asistente"
            context += f"{role}: {msg['content']}\n"
        
        return context
    
    def _should_use_tool(self, query: str) -> tuple[bool, str, str]:
        """Determina si debe usar una herramienta considerando el contexto"""
        conversation_context = self._get_conversation_context()
        
        decision_prompt = f"""Eres un asistente que decide qué herramientas usar basándote en la conversación.

{conversation_context}

Herramientas disponibles:
{self._format_tools()}

Nueva pregunta del usuario: {query}

IMPORTANTE:
- Si es small-talk (saludos, agradecimientos), NO uses herramientas
- Si necesitas información de Netflix, usa las herramientas apropiadas
- Considera el contexto de la conversación anterior

Responde SOLO con JSON válido:
{{"use_tool": true, "tool_name": "nombre_herramienta", "tool_input": "input"}}
O:
{{"use_tool": false, "response": "tu respuesta"}}
"""
        
        try:
            response = self.llm.invoke([HumanMessage(content=decision_prompt)])
            response_text = response.content
            
            if "{" in response_text and "}" in response_text:
                json_start = response_text.find("{")
                json_end = response_text.rfind("}") + 1
                json_str = response_text[json_start:json_end]
                decision = json.loads(json_str)
                
                if decision.get("use_tool"):
                    return True, decision.get("tool_name"), decision.get("tool_input")
                else:
                    return False, "", decision.get("response", "")
        except Exception as e:
            print(f"Error al decidir herramienta: {e}")
            return False, "", ""
        
        return False, "", ""
    
    def run(self, query: str) -> str:
        """Ejecuta el agente con memoria conversacional"""
        # Agrega la consulta al historial
        self.conversation_history.append({"role": "user", "content": query})
        
        # Decide si usar herramienta
        use_tool, tool_name, tool_input = self._should_use_tool(query)
        
        if use_tool and tool_name in self.tools:
            print(f"Usando: {tool_name}({tool_input})")
            
            # Ejecuta herramienta
            tool_result = self.tools[tool_name].invoke(tool_input)
            
            # Genera respuesta con contexto conversacional
            context = self._get_conversation_context()
            final_prompt = f"""Eres un asistente experto en Netflix. Responde de manera natural y conversacional.

{context}

Nueva pregunta: {query}

Información obtenida:
{tool_result}

Genera una respuesta natural y útil. Si el usuario preguntó algo relacionado con conversaciones anteriores, refiérelo apropiadamente."""
            
            response = self.llm.invoke([HumanMessage(content=final_prompt)])
            final_answer = response.content
            
        else:
            print(f"Conversación directa")
            context = self._get_conversation_context()
            
            response = self.llm.invoke([
                SystemMessage(content=f"""Eres un asistente amigable sobre Netflix. Mantén una conversación natural.

{context}

Responde considerando el contexto de la conversación."""),
                HumanMessage(content=query)
            ])
            final_answer = response.content
        
        # Guardar respuesta en historial
        self.conversation_history.append({"role": "assistant", "content": final_answer})
        
        return final_answer
    
    def clear_history(self):
        """Limpia el historial de conversación"""
        self.conversation_history = []
        print("Historial limpiado")
    
    def get_history(self):
        """Obtiene el historial completo"""
        return self.conversation_history

agent_memoria = AgenticRAGWithMemory(llm, tools, embeddings, vectorstore, max_history=5)

print("Agente creado.")

Agente creado.


Pruebas sobre el agente

In [ ]:
### Prueba de memoria con conversacion con contexto

print("PRUEBA DE MEMORIA CONVERSACIONAL")

print("\n1. Primera consulta:")
r1 = agent_memoria.run("Busca películas de terror")
print(f"Usuario: Busca películas de terror")
print(f"Agente: {r1}")

# Pregunta de seguimiento donde deberia recordar "peliculas de terror"
print("\n2. Pregunta de seguimiento:")
r2 = agent_memoria.run("¿Cuál de esas es la más reciente?")
print(f"Usuario: ¿Cuál de esas es la más reciente?")
print(f"Agente: {r2}")

print("\n3. Cambio de tema:")
r3 = agent_memoria.run("Ahora busca comedias románticas")
print(f"Usuario: Ahora busca comedias románticas")
print(f"Agente: {r3}")

# Referencia a conversación anterior
print("\n4. Referencia anterior:")
r4 = agent_memoria.run("Prefiero las películas de terror que me mostraste antes")
print(f"Usuario: Prefiero las películas de terror que me mostraste antes")
print(f"Agente: {r4}")

PRUEBA DE MEMORIA CONVERSACIONAL

1. Primera consulta:
Usando: search_netflix(género: terror)
Usuario: Busca películas de terror
Agente: ¡Claro! Pero hay un detalle importante: de los resultados que mencionaste, **Rogue Warfare (2019)** es la única película de terror en la lista. Las otras dos, *Oklahoma City* y *E-Team*, son documentales, así que no cumplen con el género de terror que estabas buscando.

Si quieres películas de terror más en carne y hueso —como las de sobrenaturales, psicológico, o misterio— puedo recomendarte algunas verdaderamente impactantes de Netflix. ¿Te gustaría algo con un toque de humedad, por ejemplo un clásico de terror psicológico como *The Babadook* o *The Conjuring*? O tal vez algo más urbano y moderno como *A Quiet Place* o *It Follows*?

¿Qué tipo de terror te parece más atractivo? 😊

2. Pregunta de seguimiento:
Usando: search_netflix(terror, recent)
Usuario: ¿Cuál de esas es la más reciente?
Agente: ¡Buena pregunta! Si estamos hablando de las opciones 

### Cargamos los datos de los papers

In [12]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.schema import Document
import os

pdf_folder = "./Datos Obligatorios"

if os.path.exists(pdf_folder):
    print(f"Carpeta encontrada: {pdf_folder}")
    archivos = os.listdir(pdf_folder)
    pdfs = [f for f in archivos if f.endswith('.pdf')]
    print(f"PDFs encontrados: {len(pdfs)}")
    for pdf in pdfs:
        print(f"   • {pdf}")
else:
    print(f"Carpeta no encontrada: {pdf_folder}")

print("\nCargando PDFs...")

try:
    loader = DirectoryLoader(
        pdf_folder,
        glob="**/*.pdf",
        loader_cls=PyPDFLoader,
        show_progress=True  # Mostrar progreso
    )
    papers_raw = loader.load()
    print(f"\nPDFs cargados con extio.")
    print(f"   Total de páginas: {len(papers_raw)}")
    
    # Mostramos informacion de los documentos cargados
    print("\nResumen de documentos cargados:")
    sources = {}
    for doc in papers_raw:
        source = doc.metadata.get('source', 'Unknown')
        filename = os.path.basename(source)
        if filename not in sources:
            sources[filename] = 0
        sources[filename] += 1
    
    for filename, pages in sources.items():
        print(f"   • {filename}: {pages} páginas")
        
except Exception as e:
    print(f"Error al cargar PDFs: {e}")
    
    papers_raw = create_sample_papers()

Carpeta encontrada: ./Datos Obligatorios
PDFs encontrados: 6
   • 2507.14805v1-1.pdf
   • 2507.14805v1.pdf
   • Attention is all you need.pdf
   • GPT Improving Language Understanding by Generative Pretraining.pdf
   • Language Models are Few-Shot Learners (GPT3).pdf
   • Scaling Laws for Neural Language Models.pdf

Cargando PDFs...


100%|██████████| 6/6 [00:04<00:00,  1.41it/s]


PDFs cargados con extio.
   Total de páginas: 200

Resumen de documentos cargados:
   • 2507.14805v1-1.pdf: 34 páginas
   • 2507.14805v1.pdf: 34 páginas
   • Attention is all you need.pdf: 15 páginas
   • GPT Improving Language Understanding by Generative Pretraining.pdf: 12 páginas
   • Language Models are Few-Shot Learners (GPT3).pdf: 75 páginas
   • Scaling Laws for Neural Language Models.pdf: 30 páginas


### Aplicamos chunking a los PDFs cargados

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

print("Aplicando chunking...")
papers_chunks = text_splitter.split_documents(papers_raw)

# Mostramos datos de los chunks generados
print(f"\n Chunking completado:")
print(f"   Documentos originales: {len(papers_raw)} páginas")
print(f"   Chunks generados: {len(papers_chunks)} fragmentos")
print(f"   Promedio: {len(papers_chunks)/len(papers_raw):.1f} chunks por página")

Aplicando chunking...

 Chunking completado:
   Documentos originales: 200 páginas
   Chunks generados: 928 fragmentos
   Promedio: 4.6 chunks por página


### Creamos base de datos vectorial para los papers

In [ ]:
from langchain_community.vectorstores import Chroma

print("Creando base vectorial de papers...")
print(f"\nProcesando {len(papers_chunks)} chunks")

papers_vectorstore = Chroma.from_documents(
    documents=papers_chunks,
    embedding=embeddings,
    persist_directory="./papers_vectordb",
    collection_name="academic_papers"
)

print("\nBase vectorial de papers creada exitosamente!")


 Creando base vectorial de papers...

Procesando 928 chunks

Base vectorial de papers creada exitosamente!


### Creamos nuevas herramientas para los papers

In [15]:
@tool
def search_papers(query: str) -> str:
    """Busca información en los papers academicos cargados.
    
    Args:
        query: La pregunta o tema a buscar en los papers académicos
    
    Returns:
        Fragmentos relevantes de los papers con la información solicitada.
    """
    try:
        docs = papers_retriever.invoke(query)
        if not docs:
            return "No se encontró información sobre ese tema en los papers disponibles."
        
        results = []
        for i, doc in enumerate(docs[:3], 1):
            source_file = os.path.basename(doc.metadata.get('source', 'Unknown'))
            page = doc.metadata.get('page', 'N/A')
            content = doc.page_content[:500] + "..." if len(doc.page_content) > 500 else doc.page_content
            
            results.append(f"[{i}] {source_file} (página {page}):\n{content}")
        
        return "\n\n".join(results)
    except Exception as e:
        return f"Error al buscar en papers: {str(e)}"

@tool
def summarize_paper_topic(topic: str) -> str:
    """Resume información sobre un tema específico encontrado en los papers (ej: transformers, GPT, attention, scaling laws).
    
    Args:
        topic: El tema o concepto a resumir
    
    Returns:
        Resumen compilado de múltiples fragmentos sobre el tema.
    """
    try:
        docs = papers_vectorstore.similarity_search(topic, k=5)
        
        if not docs:
            return f"No se encontró información sobre '{topic}' en los papers."
        
        # Agrupar por fuente
        by_source = {}
        for doc in docs:
            source = os.path.basename(doc.metadata.get('source', 'Unknown'))
            if source not in by_source:
                by_source[source] = []
            by_source[source].append(doc.page_content)
        
        # Crear resumen
        summary = f"Información sobre '{topic}' encontrada en {len(by_source)} documentos:\n\n"
        
        for source, contents in by_source.items():
            combined = "\n".join(contents)[:600]
            summary += f"📄 {source}:\n{combined}...\n\n"
        
        return summary
    except Exception as e:
        return f"Error al resumir: {str(e)}"

tools_completo = [
    search_netflix,
    get_movie_details,
    count_titles_by_type,
    search_papers,
    summarize_paper_topic
]

### Creamos el agente RAG con las nuevas herramientas

In [16]:
# CREAMOS EL AGENTE RAG CON LAS NUEVAS HERRAMIENTAS

agent_completo = AgenticRAGWithMemory(
    llm=llm,
    tools=tools_completo,
    embeddings=embeddings,
    vectorstore=vectorstore,
    max_history=5
)

### Pruebas finales

In [17]:
print("PRUEBAS DEL SISTEMA COMPLETO\n")

print("Pregutna de Attention Mechanism")
r1 = agent_completo.run("Explícame qué es el mecanismo de atención según el paper Attention is All You Need")
print(f"Usuario: Explícame qué es el mecanismo de atención según el paper Attention is All You Need")
print(f"Agente: {r1[:300]}...\n")

print("Pregunta de GPT")
r2 = agent_completo.run("¿Qué es GPT y cómo funciona?")
print(f"Usuario: ¿Qué es GPT y cómo funciona?")
print(f"Agente: {r2[:300]}...\n")

print("Pregunta de Netflix")
r3 = agent_completo.run("Ahora busca películas de acción")
print(f"Usuario: Ahora busca películas de acción")
print(f"Agente: {r3[:300]}...\n")

PRUEBAS DEL SISTEMA COMPLETO

Pregutna de Attention Mechanism
Usando: search_papers(Attention is All You Need)
Usuario: Explícame qué es el mecanismo de atención según el paper Attention is All You Need
Agente: ¡Claro! Vamos a explicarlo de forma sencilla y natural, como si estuviéramos hablando en una conversación.

El mecanismo de atención (o *attention mechanism*) que se presenta en el paper *"Attention is All You Need"* —el que marcó un gran salto en el desarrollo de modelos de lenguaje— es como una fo...

Pregunta de GPT
Usando: summarize_paper_topic(GPT)
Usuario: ¿Qué es GPT y cómo funciona?
Agente: ¡Claro! Vamos a hablar de GPT, y esta vez lo hago de forma sencilla, natural y como si estuviéramos hablando en la cafetería —sin tecnicismos abrumadores 😊.

---

**¿Qué es GPT?**  
GPT es una serie de modelos de lenguaje muy avanzados, desarrollados por OpenAI, que aprenden a entender, generar y co...

Pregunta de Netflix
Usando: search_netflix(género: acción)
Usuario: Ahora busca pe

### Control de alucinaciones

In [18]:
from langchain_core.messages import HumanMessage, SystemMessage

class AgenticRAGWithHallucinationControl(AgenticRAGWithMemory):
    """Agente mejorado con control de alucinaciones"""
    
    def __init__(self, llm, tools, embeddings, vectorstore, max_history=5):
        super().__init__(llm, tools, embeddings, vectorstore, max_history)
        self.hallucination_checks = 0
        self.tool_usage_count = {}
    
    def _validate_response(self, response: str, tool_result: str) -> str:
        """Valida que la respuesta esté basada en los datos proporcionados"""
        
        validation_prompt = f"""Eres un validador de respuestas. Tu trabajo es verificar si la respuesta está basada ÚNICAMENTE en la información proporcionada.

INFORMACIÓN DISPONIBLE:
{tool_result[:800]}

RESPUESTA A VALIDAR:
{response}

Analiza si la respuesta:
1. Está basada en la información proporcionada
2. No inventa datos que no están en la información
3. Admite honestamente si no tiene información suficiente

Responde SOLO con JSON:
{{"valida": true, "confianza": "alta"}} si está bien
{{"valida": false, "razon": "motivo"}} si hay problemas
"""
        
        try:
            validation = self.llm.invoke([HumanMessage(content=validation_prompt)])
            validation_text = validation.content
            
            if "false" in validation_text.lower() or "problema" in validation_text.lower():
                self.hallucination_checks += 1
                print(f"Posible alucinación detectada")
                # Regenerar respuesta con advertencia
                return "No tengo suficiente información confiable para responder esa pregunta específica basándome en los datos disponibles."
            
        except Exception as e:
            print(f"Error en validación: {e}")
        
        return response
    
    def run(self, query: str, validate: bool = True) -> str:
        """Ejecuta el agente con validación opcional de alucinaciones"""
        
        # Agregar consulta al historial
        self.conversation_history.append({"role": "user", "content": query})
        
        # Decidir si usar herramienta
        use_tool, tool_name, tool_input = self._should_use_tool(query)
        
        if use_tool and tool_name in self.tools:
            print(f"Usando: {tool_name}({tool_input})")
            self.tool_usage_count[tool_name] = self.tool_usage_count.get(tool_name, 0) + 1
            tool_result = self.tools[tool_name].invoke(tool_input)
            
            # Verifica si hay resultados
            if "no se encontr" in tool_result.lower() or "error" in tool_result.lower():
                final_answer = "Lo siento, no encontré información relevante sobre eso. ¿Puedo ayudarte con algo más?"
            else:
                # Generar respuesta con instrucciones para evitar alucinaciones
                context = self._get_conversation_context()
                final_prompt = f"""Eres un asistente honesto y preciso. Responde SOLO basándote en la información proporcionada.

{context}

Nueva pregunta: {query}

INFORMACIÓN DISPONIBLE:
{tool_result}

REGLAS IMPORTANTES:
1. Usa SOLO la información proporcionada arriba
2. Si no tienes información suficiente, admítelo honestamente
3. NO inventes datos, fechas, nombres o detalles que no estén en la información
4. Si la información es parcial, di "Según la información disponible..." o similar
5. Responde de manera natural pero precisa

Tu respuesta:"""
                
                response = self.llm.invoke([HumanMessage(content=final_prompt)])
                final_answer = response.content
                
                # Validar respuesta si esta activado
                if validate:
                    final_answer = self._validate_response(final_answer, tool_result)
            
        else:
            print(f"Conversación directa")
            context = self._get_conversation_context()
            
            response = self.llm.invoke([
        SystemMessage(content=f"""Eres un asistente especializado ÚNICAMENTE en:
1. Contenido de Netflix (películas y series)
2. Papers académicos sobre IA (Transformers, GPT, RAG, etc.)

REGLA CRÍTICA: Si te preguntan sobre algo que NO esté en estos dos temas, di honestamente:
"Lo siento, no tengo información sobre [tema]. Solo puedo ayudarte con contenido de Netflix y papers académicos sobre inteligencia artificial."

NO inventes información sobre cualquier tema fuera de tu especialización.

{context}"""),
                HumanMessage(content=query)
            ])
            final_answer = response.content
        
        # Guardar en historial
        self.conversation_history.append({"role": "assistant", "content": final_answer})
        
        return final_answer
    
    def get_stats(self):
        """Obtiene estadísticas de uso"""
        return {
            "total_conversaciones": len(self.conversation_history) // 2,
            "uso_herramientas": self.tool_usage_count,
            "checks_alucinacion": self.hallucination_checks
        }

# Agente mejorado con control de alucinaciones ademas de la memoria
agent_final = AgenticRAGWithHallucinationControl(
    llm=llm,
    tools=tools_completo,
    embeddings=embeddings,
    vectorstore=vectorstore,
    max_history=5
)

### Pruebas de control de alucinaciones

In [19]:
r2 = agent_final.run("¿Cuáles son los personajes principales de “Cien años de soledad”?")
print(f"Usuario: ¿Cuáles son los personajes principales de “Cien años de soledad”?")
print(f"Agente: {r2}\n")

r3 = agent_final.run("¿Qué ingredientes se necesitan para preparar una lasaña vegetariana?")
print(f"Usuario: ¿Qué ingredientes se necesitan para preparar una lasaña vegetariana?")
print(f"Agente: {r3}\n")

r4 = agent_final.run("Resumir el artículo “Attention Is All You Need” en tres párrafos.")
print(f"Usuario: Resumir el artículo “Attention Is All You Need” en tres párrafos.")
print(f"Agente: {r4}\n")

r5 = agent_final.run("¿Qué películas de ciencia ficción ganaron un premio Oscar en los últimos cinco años?")
print(f"Usuario: ¿Qué películas de ciencia ficción ganaron un premio Oscar en los últimos cinco años?")
print(f"Agente: {r5}\n")

r6 = agent_final.run("Como estas?")
print(f"Usuario: Como estas?")
print(f"Agente: {r6}\n")

Conversación directa
Usuario: ¿Cuáles son los personajes principales de “Cien años de soledad”?
Agente: Lo siento, no tengo información sobre "Cien años de soledad". Solo puedo ayudarte con contenido de Netflix y papers académicos sobre inteligencia artificial.

Conversación directa
Usuario: ¿Qué ingredientes se necesitan para preparar una lasaña vegetariana?
Agente: Lo siento, no tengo información sobre cómo preparar una lasaña vegetariana. Solo puedo ayudarte con contenido de Netflix y papers académicos sobre inteligencia artificial.

Usando: summarize_paper_topic(Attention Is All You Need)
Usuario: Resumir el artículo “Attention Is All You Need” en tres párrafos.
Agente: Según la información disponible, no se puede resumir el artículo “Attention Is All You Need” en tres párrafos debido a que el contenido proporcionado es insuficiente y repetitivo, solo incluyendo frases como "Note that the attentions are very sharp for this word" sin contexto, desarrollo ni detalles relevantes sobre

### Agente final

In [21]:
req = agent_final.run("Ingresar consulta")
print(req)
# Continuar para probar el agente

Conversación directa
¡Claro! Por favor, ingresa tu consulta. Estoy aquí para ayudarte con:

1. **Contenido de Netflix** (películas, series, recomendaciones, etc.)  
2. **Papers académicos sobre inteligencia artificial** (Transformers, GPT, RAG, modelos de lenguaje, arquitecturas de IA, etc.)

Si tienes dudas sobre alguno de estos temas, solo dime.  

¿Qué necesitas saber hoy? 🚀🎬🧠
